# stacks-bench benchmark dashboard (local)

This notebook provides a lightweight Metabase-like explorer for `stacks-bench` results stored in SQLite.

**What you get**
- A list of recent runs.
- Filters for time range (absolute) and network.
- A run selector dropdown.
- A few starter summary tables.

**DB path**
By default this uses:
- `../target/release/.stacks-bench/appdata/stacks-bench.db`

You can override it in Cell 2.

In [3]:
%pip install ipywidgets pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 914.9/914.9 kB 8.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 10.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 9.6 MB/s  0:00:01eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 10.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [ipywidgets]7 [ipywidgets]
Note: you may need to restart the kernel to use updated packages.


In [12]:
from __future__ import annotations

import os
import sqlite3
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

import pandas as pd

# Optional UI deps.
try:
    import ipywidgets as widgets
    from IPython.display import display, Markdown, clear_output
except Exception as e:
    widgets = None
    display = None
    Markdown = None
    clear_output = None
    print("ipywidgets not available yet. If needed, run: pip install ipywidgets")

DB_PATH = Path(os.environ.get(
    "STACKS_BENCH_DB",
    str(Path.cwd().parent / "../target/release/.stacks-bench/appdata/stacks-bench.db"),
)).resolve()

print("DB:", DB_PATH)
print("Exists:", DB_PATH.exists())

DB: /Users/cylwit/Code/github.com/stacks-network/stacks-core/target/release/.stacks-bench/appdata/stacks-bench.db
Exists: True


In [13]:
def query_df(db_path: Path, sql: str, params: Tuple[Any, ...] = ()) -> pd.DataFrame:
    con = sqlite3.connect(str(db_path))
    try:
        return pd.read_sql_query(sql, con, params=params)
    finally:
        con.close()


def list_networks(db_path: Path) -> pd.DataFrame:
    return query_df(
        db_path,
        """
        SELECT n.id AS network_id, n.name AS network
        FROM network n
        ORDER BY n.id
        """,
    )


def list_recent_runs(
    db_path: Path,
    network_name: Optional[str] = None,
    start_ts: Optional[str] = None,
    end_ts: Optional[str] = None,
    limit: int = 50,
) -> pd.DataFrame:
    # start_ts/end_ts should be ISO strings or None.
    params: List[Any] = []
    where: List[str] = []

    if network_name:
        where.append("n.name = ?")
        params.append(network_name)

    if start_ts:
        where.append("br.start_time >= ?")
        params.append(start_ts)

    if end_ts:
        where.append("br.start_time <= ?")
        params.append(end_ts)

    where_sql = ("WHERE " + " AND ".join(where)) if where else ""

    # Join: benchmark_run -> chainstate -> network.
    sql = f"""
        SELECT
            br.id AS run_id,
            br.run_name,
            n.name AS network,
            br.start_time,
            br.end_time,
            length(br.args_json) AS args_len,
            hex(br.git_commit_hash) AS git_commit_hash_hex
        FROM benchmark_run br
        JOIN chainstate cs ON cs.id = br.chainstate_id
        JOIN network n ON n.id = cs.network_id
        {where_sql}
        ORDER BY br.start_time DESC
        LIMIT ?
    """

    params.append(int(limit))
    return query_df(db_path, sql, tuple(params))


# Quick smoke check (non-fatal if DB not present yet)
if DB_PATH.exists():
    display(list_networks(DB_PATH).head(10) if display else list_networks(DB_PATH).head(10))
    display(list_recent_runs(DB_PATH, limit=5) if display else list_recent_runs(DB_PATH, limit=5))

,network_id,network
0,1,mainnet
1,2,testnet
2,3,regtest


,run_id,run_name,network,start_time,end_time,args_len,git_commit_hash_hex
0,3,20260112-202531,mainnet,2026-01-12 20:25:31.542111,None,146,C868B35C0888AD1906C9CD59AC5637BC256FC3DD
1,2,20260112-164331,mainnet,2026-01-12 16:43:31.498092,2026-01-12 17:10:25.255549,146,C868B35C0888AD1906C9CD59AC5637BC256FC3DD
2,1,20260112-144658,mainnet,2026-01-12 14:46:58.306714,2026-01-12 15:09:01.723146,145,C868B35C0888AD1906C9CD59AC5637BC256FC3DD


In [14]:
if widgets is None:
    raise RuntimeError(
        "ipywidgets is required for the interactive dashboard. "
        "Install it with: pip install ipywidgets"
    )

networks_df = list_networks(DB_PATH)
network_options = ["(any)"] + networks_df["network"].tolist()

network_dd = widgets.Dropdown(options=network_options, value="(any)", description="Network:")

start_picker = widgets.DatetimePicker(
    description="Start >=",
    value=None,
    disabled=False,
)
end_picker = widgets.DatetimePicker(
    description="Start <=",
    value=None,
    disabled=False,
)

limit_slider = widgets.IntSlider(
    value=50,
    min=10,
    max=500,
    step=10,
    description="Runs:",
    continuous_update=False,
)

run_dd = widgets.Dropdown(options=[], description="Run:")
refresh_btn = widgets.Button(description="Refresh runs", button_style="primary")

runs_out = widgets.Output()
summary_out = widgets.Output()

controls = widgets.HBox([
    network_dd,
    start_picker,
    end_picker,
    limit_slider,
    refresh_btn,
])

display(controls)
display(run_dd)
display(runs_out)
display(summary_out)

Dropdown(description='Run:', options=(), value=None)

Output()

Output()

In [ ]:
def _to_iso(dt: Optional[datetime]) -> Optional[str]:
    if dt is None:
        return None
    # SQLite stores timestamps via Diesel; treat as naive/ISO.
    return dt.replace(tzinfo=None).isoformat(sep=" ")


def refresh_runs(*_args: Any) -> None:
    with runs_out:
        clear_output(wait=True)
        net = None if network_dd.value == "(any)" else network_dd.value
        runs_df = list_recent_runs(
            DB_PATH,
            network_name=net,
            start_ts=_to_iso(start_picker.value),
            end_ts=_to_iso(end_picker.value),
            limit=int(limit_slider.value),
        )
        if runs_df.empty:
            display(Markdown("No runs found for the current filters."))
            run_dd.options = []
            return

        # Make a nice label
        runs_df["label"] = (
            runs_df["run_id"].astype(str)
            + " | "
            + runs_df["network"].astype(str)
            + " | "
            + runs_df["start_time"].astype(str)
            + " | "
            + runs_df["run_name"].fillna("")
        )

        # Display table
        display(runs_df[["run_id", "network", "start_time", "end_time", "run_name"]].head(50))

        # Populate dropdown
        options = list(zip(runs_df["label"].tolist(), runs_df["run_id"].tolist()))
        run_dd.options = options
        run_dd.value = runs_df["run_id"].iloc[0]


def run_summary(db_path: Path, run_id: int) -> Dict[str, pd.DataFrame]:
    out: Dict[str, pd.DataFrame] = {}

    out["run"] = query_df(
        db_path,
        """
        SELECT
            br.id AS run_id,
            br.run_name,
            n.name AS network,
            br.start_time,
            br.end_time,
            br.args_json
        FROM benchmark_run br
        JOIN chainstate cs ON cs.id = br.chainstate_id
        JOIN network n ON n.id = cs.network_id
        WHERE br.id = ?
        """,
        (int(run_id),),
    )

    out["blocks"] = query_df(
        db_path,
        """
        SELECT
            count(*) AS block_rows,
            sum(CASE WHEN synthetic_block_id IS NULL THEN 1 ELSE 0 END) AS real_block_rows,
            sum(CASE WHEN synthetic_block_id IS NOT NULL THEN 1 ELSE 0 END) AS synthetic_block_rows,
            avg(total_duration_us)/1000.0 AS avg_total_ms,
            avg(setup_duration_us)/1000.0 AS avg_setup_ms,
            avg(execution_duration_us)/1000.0 AS avg_exec_ms,
            avg(commit_duration_us)/1000.0 AS avg_commit_ms
        FROM stacks_block_stats
        WHERE benchmark_run_id = ?
        """,
        (int(run_id),),
    )

    out["txs"] = query_df(
        db_path,
        """
        SELECT
            count(*) AS tx_rows,
            avg(duration_us)/1000.0 AS avg_tx_ms,
            percentile_cont(0.5) WITHIN GROUP (ORDER BY duration_us) / 1000.0 AS p50_tx_ms,
            percentile_cont(0.95) WITHIN GROUP (ORDER BY duration_us) / 1000.0 AS p95_tx_ms
        FROM stacks_tx_stats
        WHERE benchmark_run_id = ?
        """,
        (int(run_id),),
    )

    return out


def on_run_change(change: Dict[str, Any]) -> None:
    if change.get("name") != "value":
        return
    run_id = change["new"]
    with summary_out:
        clear_output(wait=True)
        s = run_summary(DB_PATH, int(run_id))
        display(Markdown(f"## Run {run_id}"))
        display(Markdown("### Run metadata"))
        display(s["run"])
        display(Markdown("### Block stats (aggregate)"))
        display(s["blocks"])
        display(Markdown("### Tx stats (aggregate)"))
        display(s["txs"])


refresh_btn.on_click(refresh_runs)
run_dd.observe(on_run_change)

# Initial load
refresh_runs()